# 1. Introduction to Model Implementation

### Overview

In this notebook, we implement the autoencoder model. Before training, however, the preprocessed data undergoes a few final transformations. The dataset is divided into three subsets: a training set, a test set, and a final subset that will later be used to simulate a real-time stream of incoming network flows, which will be classified as either benign or anomalous.

### Time-based fragmentation

In the CIC-DDoS2019 dataset, each record contains a timestamp indicating the exact time at which the corresponding network flow was captured. Each row represents a complete network flow, which corresponds to a single connection.

To preserve the temporal characteristics of a real-world deployment, the training and test sets are created from  continuous time intervals. A separate, later time interval is reserved to simulate future "real-time" traffic. Consequently, these two time windows are completely disjoint, preventing any temporal overlap between the training/testing data and the simulated production data.

### Training with benign data

Since the selected model is an autoencoder, it is trained exclusively on benign traffic. This allows the model to learn the normal patterns of network behavior. During inference, any incoming flow is reconstructed based on these learned patterns, so anomalous or attack traffic is expected to produce a significantly higher reconstruction error than benign traffic.

To validate this approach, the temporal distribution of benign and attack flows is first analyzed across the entire observation period.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from IPython.display import display

df = pd.read_csv("./data/pre_processed/CIC-DDoS2019v2.csv")

display(df['timestamp'])
print(df['label'].unique())

In [ ]:
def standarize_timestamp(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df.sort_values(by='timestamp', inplace=True)

standarize_timestamp(df)

display(df['timestamp'])

In [ ]:
rows_by_time = df.groupby('timestamp').size()
plt.figure(figsize=(12, 8))
rows_by_time.plot(kind='line', color='skyblue')
plt.xlabel('Timestamp')
plt.ylabel('Number of Rows')
plt.title('Number of Rows per Timestamp')
plt.grid(axis='y')
plt.show()

In [ ]:
df['label'] = np.where(df['label'] == 'BENIGN', 0, 1)

flows_by_time_benign = df[df['label'] == 0].groupby('timestamp').size()
flows_by_time_attack = df[df['label'] != 0].groupby('timestamp').size()
plt.figure(figsize=(12, 8))
plt.plot(flows_by_time_attack.index, flows_by_time_attack.values, marker='o', linestyle='-', label='DDoS Attack')
plt.plot(flows_by_time_benign.index, flows_by_time_benign.values, marker='o', linestyle='-', label='Benign')
plt.title('Benign/DDoS Flows Through Time')

plt.xlabel('Time')
plt.ylabel('Count')
plt.grid(True)

ax = plt.gca()
ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=5))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.legend()
plt.show()



### Data Splits

We can see in the graph above that the DDoS attacks are concentrated within a fixed time window, specifically between 03:49 and 04:19. Any flow record outside this time interval represents a benign connection.

To create a realistic training environment, we use data from both ends of the observation period: the first 20 minutes and the last 20 minutes, where no DDoS attacks are present. This also helps introduce some variability into the training data in case the timestamp exhibits time-dependent connection patterns.

The remaining data will be used to simulate a real-time stream of network flows, preserving the temporal order of the original traffic. Maintaining this time correlation is important to accurately emulate a continuous production environment.

As shown below, we have more than 31,000 flow records available for training the autoencoder. Since the dataset contains approximately 30 features, this provides more than enough data for training a relatively small autoencoder model.

In [ ]:
first_records = df[df['timestamp'] <= pd.Timestamp('2017-07-07 03:52')].shape[0]
last_records = df[df['timestamp'] >= pd.Timestamp('2017-07-07 04:40')].shape[0]

print(first_records)
print(last_records)


In [ ]:
k = 15000
l = 25000
df_tails = pd.concat([
    df.iloc[:k],
    df.iloc[-l:]
])

df_center = df.iloc[k:-l]

display(df_tails['label'].nunique())
display(df_center['label'].nunique())

display(df_tails.head(3))

### Training Set

We now have a new subset containing the tails of the original dataset, which consists exclusively of benign network flows. This subset contains 40,000 records: 15,000 from the beginning of the dataset and the remaining 25,000 from the end, as the original data is ordered by timestamp. The remaining portion of the dataset will be used later, once the complete system has been implemented.

The next step is to preprocess this subset by performing several cleaning operations, such as removing the label and timestamp columns and standardizing the feature values. The processed data will then be split into two additional subsets: a training/testing subset containing 30,000 records and a validation subset containing the remaining 10,000 records, which will not be used during training.